In [1]:
import numpy as np
import anafibre as fib


# Create a step-index fibre
fibre = fib.StepIndexFibre(
    core_radius=250e-9,
    n_core=2.00,
    n_clad=1.33
)

# Get modes at 700 nm
wl = 700e-9
modes = []

ell_max = fibre.ell_max(wavelength=wl)
for ell in range(0, ell_max + 1):
    if ell == 0:
        m_max_te = fibre.m_max(wavelength=wl, ell=ell, mode_type="TE")
        m_max_tm = fibre.m_max(wavelength=wl, ell=ell, mode_type="TM")
        for m in range(1, m_max_te + 1):
            modes.append(fibre.TE(n=m, wl=wl))
        for m in range(1, m_max_tm + 1):
            modes.append(fibre.TM(n=m, wl=wl))
    else:
        m_max = fibre.m_max(wavelength=wl, ell=ell)
        for m in range(1, m_max + 1):
            if m % 2 == 1:
                modes.append(fibre.HE(ell=ell, n=m, wl=wl))
            else:
                modes.append(fibre.EH(ell=ell, n=m, wl=wl))

modes = [m for m in modes if m is not None]
modes = sorted(modes, key=lambda m: m.neff, reverse=True)
fib.display_modes(*modes)

# Calculate field distributions
mode = modes[0]  # Use the first mode from the list
x = np.linspace(-2*fibre.core_radius, 2*fibre.core_radius, 100)
y = np.linspace(-2*fibre.core_radius, 2*fibre.core_radius, 100)
X, Y = np.meshgrid(x, y)

E = mode.E(x=X, y=Y)  # Electric field
H = mode.H(x=X, y=Y)  # Magnetic field

Mode,λ [nm],V,neff,S0,"(S1, S2, S3) / S0"
HE11,700.00,3.35,1.7955,1.00,"( 0.00, 0.00, 1.00)"
TM01,700.00,3.35,1.5500,1.00,"( 1.00, 0.00, 0.00)"
TE01,700.00,3.35,1.4762,1.00,"(-1.00, 0.00, 0.00)"
HE21,700.00,3.35,1.4571,1.00,"( 0.00, 0.00, 1.00)"


In [ ]:
anim = fib.animate_fields_xy(modes=modes[0], show=("E",),figsize=(5,5))
fib.display_anim(anim)
# anim.save("mode_animation.mp4", writer="ffmpeg", fps=30)